# Drug Target Assessment Agent — E2E Test

Interactive notebook for testing the Drug Target Assessment Agent (`src/agents/drug_target/`).

- **Cell 2**: Setup & mock data
- **Cell 3**: Tool-level — Evidence Loader
- **Cell 4**: Tool-level — External APIs (network required)
- **Cell 5**: Rubric tests
- **Cell 6**: Agent creation
- **Cell 7**: Full E2E run (LLM API key required)
- **Cell 8**: Cleanup

In [1]:
"""Setup & Imports"""
import os

# --- Disable LangSmith tracing BEFORE any langchain imports ---
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"
import logging
logging.getLogger("langsmith").setLevel(logging.ERROR)

import json
import sys
import tempfile
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "tests":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load .env
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
    print("[OK] .env loaded")
except ImportError:
    print("[WARN] python-dotenv not installed, skipping .env")

# ---------------------------------------------------------------------------
# Mock data (from test_drug_target.py)
# ---------------------------------------------------------------------------
MOCK_LR_CANDIDATES = {
    "disease": "breast cancer",
    "disease_context": "Triple-negative breast cancer",
    "pairs": [
        {"ligand": "TGFB1", "receptor": "TGFBR2", "score": 0.95},
        {"ligand": "CDH1", "receptor": "ITGB1", "score": 0.92},
        {"ligand": "SPP1", "receptor": "CD44", "score": 0.90},
        {"ligand": "VEGFA", "receptor": "FLT1", "score": 0.88},
        {"ligand": "WNT5A", "receptor": "FZD7", "score": 0.85},
        {"ligand": "CXCL12", "receptor": "CXCR4", "score": 0.83},
        {"ligand": "HGF", "receptor": "MET", "score": 0.80},
        {"ligand": "FGF2", "receptor": "FGFR1", "score": 0.78},
        {"ligand": "PDGFB", "receptor": "PDGFRB", "score": 0.76},
        {"ligand": "BMP2", "receptor": "BMPR1A", "score": 0.74},
        {"ligand": "EFNA1", "receptor": "EPHA2", "score": 0.72},
        {"ligand": "JAG1", "receptor": "NOTCH1", "score": 0.70},
        {"ligand": "DLL4", "receptor": "NOTCH3", "score": 0.68},
        {"ligand": "GDF15", "receptor": "GFRAL", "score": 0.65},
        {"ligand": "TNFSF10", "receptor": "TNFRSF10B", "score": 0.62},
    ],
}

MOCK_SELECTED_TOP3 = [
    {"ligand": "TGFB1", "receptor": "TGFBR2"},
    {"ligand": "SPP1", "receptor": "CD44"},
    {"ligand": "VEGFA", "receptor": "FLT1"},
]

# Create tmp directory with mock data
TMP_DIR = Path(tempfile.mkdtemp(prefix="drug_target_e2e_"))
(TMP_DIR / "lr_candidates.json").write_text(json.dumps(MOCK_LR_CANDIDATES, indent=2))
(TMP_DIR / "selected_top3_lr.json").write_text(json.dumps(MOCK_SELECTED_TOP3, indent=2))

print(f"[OK] TMP_DIR: {TMP_DIR}")
print(f"[OK] lr_candidates.json: {len(MOCK_LR_CANDIDATES['pairs'])} pairs")
print(f"[OK] selected_top3_lr.json: {len(MOCK_SELECTED_TOP3)} pairs")

[OK] .env loaded
[OK] TMP_DIR: /var/folders/fq/jq53ybws2glbsnkk4k3cj_lw0000gt/T/drug_target_e2e_vguz73hg
[OK] lr_candidates.json: 15 pairs
[OK] selected_top3_lr.json: 3 pairs


## 1. Tool-Level Test: Evidence Loader

Tests `load_lr_candidates` with mock data — no network required.

In [2]:
from src.agents.drug_target.tools.evidence_loader import load_lr_candidates, load_evidence_files, save_evidence

raw = load_lr_candidates.invoke({"data_dir": str(TMP_DIR)})
result = json.loads(raw)

print(f"Gene count:        {len(result['gene_symbols'])} (expect 30)")
print(f"L-R pairs:         {len(result['lr_pairs'])} (expect 15)")
print(f"Disease:           {result['disease']}")
print(f"Selected top3:     {result['selected_top3_genes']}")
print(f"Errors:            {result['errors']}")

# Verify is_selected_top3 flags
details = {d["symbol"]: d["is_selected_top3"] for d in result["gene_details"]}
print(f"\nTGFB1 is_top3:     {details['TGFB1']} (expect True)")
print(f"FGF2  is_top3:     {details['FGF2']} (expect False)")

assert len(result["gene_symbols"]) == 30
assert len(result["lr_pairs"]) == 15
assert result["disease"] == "breast cancer"
assert details["TGFB1"] is True
assert details["FGF2"] is False
print("\n✅ Evidence Loader: all assertions passed")

Gene count:        30 (expect 30)
L-R pairs:         15 (expect 15)
Disease:           breast cancer
Selected top3:     ['CD44', 'FLT1', 'SPP1', 'TGFB1', 'TGFBR2', 'VEGFA']
Errors:            []

TGFB1 is_top3:     True (expect True)
FGF2  is_top3:     False (expect False)

✅ Evidence Loader: all assertions passed


## 2. Tool-Level Tests: External APIs

Tests real API calls to OpenTargets, GWAS Catalog, and ClinicalTrials.gov.  
**Requires network access** — failures indicate API connectivity issues, not code bugs.

In [3]:
from src.agents.drug_target.tools.opentargets_tools import query_ot_target_info
from src.agents.drug_target.tools.gwas_tools import query_gwas_trait_associations
from src.agents.drug_target.tools.clinical_trials_tools import search_clinical_trials

# --- OpenTargets ---
print("=" * 60)
print("OpenTargets: query_ot_target_info('TGFB1', 'breast cancer')")
print("=" * 60)
try:
    raw_ot = query_ot_target_info.invoke({"gene_symbol": "TGFB1", "disease": "breast cancer"})
    result_ot = json.loads(raw_ot)
    print(f"Ensembl ID:      {result_ot.get('ensembl_id')}")
    print(f"Symbol:          {result_ot.get('target_info', {}).get('approvedSymbol')}")
    print(f"Disease evidence: {len(result_ot.get('disease_evidence', []))} entries")
    print(f"Errors:          {result_ot.get('errors', [])}")
except Exception as e:
    print(f"❌ OpenTargets failed: {e}")

# --- GWAS Catalog ---
print("\n" + "=" * 60)
print("GWAS: query_gwas_trait_associations('breast cancer', 'TGFB1,VEGFA')")
print("=" * 60)
try:
    raw_gwas = query_gwas_trait_associations.invoke({
        "disease": "breast cancer",
        "gene_symbols": "TGFB1,VEGFA",
    })
    result_gwas = json.loads(raw_gwas)
    print(f"Trait ID:          {result_gwas.get('trait_id')}")
    print(f"Associations:      {result_gwas.get('total_associations_scanned')}")
    print(f"Gene signals:      {list(result_gwas.get('gene_signals', {}).keys())}")
    print(f"Errors:            {result_gwas.get('errors', [])}")
except Exception as e:
    print(f"❌ GWAS failed: {e}")

# --- ClinicalTrials.gov ---
print("\n" + "=" * 60)
print("ClinicalTrials: search_clinical_trials(disease='breast cancer', gene_symbol='TGFB1')")
print("=" * 60)
try:
    raw_ct = search_clinical_trials.invoke({
        "disease": "breast cancer",
        "gene_symbol": "TGFB1",
    })
    result_ct = json.loads(raw_ct)
    print(f"Trial count:       {result_ct.get('trial_count')}")
    print(f"Phase distribution: {result_ct.get('phase_distribution')}")
    top = result_ct.get('top_trials', [])
    if top:
        print(f"Top trial:         {top[0].get('nct_id')} — {top[0].get('title', '')[:80]}")
    print(f"Errors:            {result_ct.get('errors', [])}")
except Exception as e:
    print(f"❌ ClinicalTrials failed: {e}")

print("\n✅ External API tests completed (check outputs above for errors)")

OpenTargets: query_ot_target_info('TGFB1', 'breast cancer')
Ensembl ID:      ENSG00000105329
Symbol:          TGFB1
Disease evidence: 100 entries
Errors:          []

GWAS: query_gwas_trait_associations('breast cancer', 'TGFB1,VEGFA')
Trait ID:          MONDO_0007254
Associations:      614
Gene signals:      ['TGFB1', 'VEGFA']
Errors:            []

ClinicalTrials: search_clinical_trials(disease='breast cancer', gene_symbol='TGFB1')
Trial count:       9
Phase distribution: {'PHASE1': 4, 'PHASE2': 3, 'PHASE3': 1, 'EARLY_PHASE1': 1}
Top trial:         NCT06026657 — Phase 1b/2 Study of Naxitamab (Danyelza), Gemcitabine and Ex Vivo Expanded Allog
Errors:            []

✅ External API tests completed (check outputs above for errors)


## 3. Rubric Tests

Tests `format_rubric_prompt`, `validate_assessment`, and `compute_track_total` — pure logic, no API calls.

In [4]:
from src.agents.drug_target.rubric import (
    format_rubric_prompt,
    validate_assessment,
    compute_track_total,
    DRUG_TARGET_RUBRIC,
)

# --- format_rubric_prompt ---
prompt_top3 = format_rubric_prompt({"gene_symbol": "TGFB1", "is_selected_top3": True})
prompt_non = format_rubric_prompt({"gene_symbol": "FGF2", "is_selected_top3": False})

print("--- Rubric Prompt (top3, first 300 chars) ---")
print(prompt_top3[:300])
print("...")

assert "FEATURED CANDIDATE" in prompt_top3
assert "FEATURED CANDIDATE" not in prompt_non
assert "VALIDATED TRACK" in prompt_top3
assert "EXPLORATORY TRACK" in prompt_top3
print("\n✅ format_rubric_prompt: OK")

# --- validate_assessment ---
valid_assessment = {
    "gene_symbol": "TGFB1",
    "is_selected_top3": True,
    "validated_track": {
        "existing_drug_evidence": {"score": 2, "rationale": "Galunisertib Phase 2"},
        "clinical_trial_activity": {"score": 2, "rationale": "Multiple active trials"},
        "mechanism_relevance": {"score": 3, "rationale": "Direct TGF-beta pathway"},
        "safety_profile": {"score": -1, "rationale": "Moderate immunosuppression risk"},
        "total": 6,
        "pass": True,
    },
    "exploratory_track": {
        "gwas_genetic_support": {"score": 1, "rationale": "Suggestive signals"},
        "tractability": {"score": 2, "rationale": "Clinical precedent"},
        "biological_plausibility": {"score": 3, "rationale": "Compelling evidence"},
        "novelty_opportunity": {"score": 1, "rationale": "Moderately explored"},
        "total": 7,
        "pass": True,
    },
    "classification": "validated_target",
    "priority": "high",
    "summary": "TGFB1 is a validated target with Phase 2 drug evidence.",
}

is_valid, errors = validate_assessment(valid_assessment)
print(f"\nValidation: valid={is_valid}, errors={errors}")
assert is_valid is True
assert errors == []
print("✅ validate_assessment: OK")

# --- compute_track_total ---
vt = compute_track_total(valid_assessment["validated_track"], "validated_track")
et = compute_track_total(valid_assessment["exploratory_track"], "exploratory_track")
print(f"\nValidated total:   {vt} (expect 6: 2+2+3+(-1))")
print(f"Exploratory total: {et} (expect 7: 1+2+3+1)")
assert vt == 6
assert et == 7
print("✅ compute_track_total: OK")

--- Rubric Prompt (top3, first 300 chars) ---
## Drug Target Assessment: TGFB1
**★ FEATURED CANDIDATE (selected_top3_lr)**

### Evidence Summary
```json
{
  "gene_symbol": "TGFB1",
  "is_selected_top3": true
}
```

### Evaluation Rubric

Evaluate this target against both tracks using the criteria below.
For each criterion, provide a score AND a
...

✅ format_rubric_prompt: OK

Validation: valid=True, errors=[]
✅ validate_assessment: OK

Validated total:   6 (expect 6: 2+2+3+(-1))
Exploratory total: 7 (expect 7: 1+2+3+1)
✅ compute_track_total: OK


## 4. Agent Creation Test

Tests `DrugTargetConfig` + `create_drug_target_agent` — verifies agent object, `data_path.txt`, and `evidence/` directory.  
**Requires LLM provider SDK** (creates LLM instance).

In [ ]:
from src.agents.drug_target.agent import DrugTargetConfig, create_drug_target_agent

config = DrugTargetConfig(
    data_dir=str(TMP_DIR),
    disease="breast cancer",
)
print(f"Provider:          {config.provider}")
print(f"Data dir:          {config.data_dir}")
print(f"Max iterations:    {config.max_iterations}")
print(f"Quality threshold: {config.quality_threshold}")

try:
    agent = create_drug_target_agent(config)
    print(f"\nAgent type:        {type(agent).__name__}")
    print(f"data_path.txt:     {(TMP_DIR / 'data_path.txt').exists()}")
    print(f"evidence/ dir:     {(TMP_DIR / 'evidence').is_dir()}")

    assert (TMP_DIR / "data_path.txt").exists()
    assert (TMP_DIR / "evidence").is_dir()
    print("\n✅ Agent creation: OK")
except Exception as e:
    print(f"\n❌ Agent creation failed: {e}")
    print("   (Check LLM_PROVIDER and API key in .env)")

## 5. Full Agent E2E Run

Runs the complete `evidence_collector → clinical_assessor → report_writer` pipeline.  
**Requires**: network + LLM API key. This may take several minutes.

In [ ]:
import uuid
from src.agents.drug_target.integration import run_drug_target_assessment

print(f"Running full E2E assessment...")
print(f"Data dir:  {TMP_DIR}")
print(f"Disease:   breast cancer")
print()

try:
    raw = await run_drug_target_assessment.ainvoke({
        "data_dir": str(TMP_DIR),
        "disease": "breast cancer",
    })
    result = json.loads(raw)

    if result.get("success"):
        drug_md = result["drug_md"]
        print(f"✅ drug.md generated ({len(drug_md)} chars)")
        print(f"   Path: {result.get('drug_md_path')}")
        print("\n" + "=" * 60)
        print("drug.md (first 2000 chars):")
        print("=" * 60)
        print(drug_md[:2000])
        if len(drug_md) > 2000:
            print(f"\n... ({len(drug_md) - 2000} more chars)")
    else:
        print(f"❌ Assessment failed: {result.get('error')}")

except Exception as e:
    print(f"❌ E2E run failed: {e}")
    print("   (Check LLM_PROVIDER and API key in .env)")

# Show evidence files
evi_dir = TMP_DIR / "evidence"
if evi_dir.exists():
    files = list(evi_dir.glob("*.json"))
    print(f"\nEvidence files ({len(files)}):")
    for f in sorted(files):
        print(f"  {f.name} ({f.stat().st_size} bytes)")

## 6. Cleanup

In [ ]:
import shutil

if TMP_DIR.exists():
    # Uncomment to actually delete:
    # shutil.rmtree(TMP_DIR)
    # print(f"✅ Cleaned up {TMP_DIR}")
    print(f"TMP_DIR preserved at: {TMP_DIR}")
    print("Uncomment shutil.rmtree(TMP_DIR) above to auto-clean.")
else:
    print("Already cleaned up.")